# 07 — Scoring and ranking

The strategy's actual output. Scores the real cross-section with the fitted ranker, then runs the full decision pipeline — validate, exclude sectors, rank — to produce a contiguous 1..N ranking of the whole surviving universe.

**Data mode: REAL DATA.** Every figure below is computed from the genuine Bloomberg CSV exports for this strategy, through production AtlasQuant code. No synthetic, simulated, imputed, or approximated value appears anywhere in this notebook.

Those exports are gitignored, so they are absent in a fresh clone. When they are missing this notebook still executes top to bottom, but every computing cell prints an explicit *real data not present — cannot produce a genuine result* notice instead of a number. It never falls back to synthetic data: a plausible-looking figure derived from invented inputs is not a result (see `CLAUDE.md`).

In [ ]:
import _real_data as rd

print(rd.banner())
AVAILABLE = rd.data_available()


## What this strategy produces

A score and a rank per instrument. Nothing else. There is no qualification threshold, no position count, no weight, no capital allocation and no fallback sleeve — every candidate surviving validation and the sector exclusion is ranked, full stop.

The score is the LambdaRank model's raw `predict` output: an **unbounded real-valued ranking margin**. Its *order* is meaningful; its magnitude is not a probability and is not calibrated. Only relative order within one scoring batch should ever be read from it.

## Fit the model for the target cycle

In [ ]:
if AVAILABLE:
    from datetime import datetime, time

    from atlas_quant.strategies.multi_factor_ranking_ml.config import (
        FEATURE_SCHEMA_VERSION,
        STRATEGY_VERSION,
    )
    from atlas_quant.strategies.multi_factor_ranking_ml.estimator import build_lgbm_ranker_estimator
    from atlas_quant.strategies.multi_factor_ranking_ml.model_training import train_model
    from atlas_quant.strategies.multi_factor_ranking_ml.training_dataset import (
        build_training_dataset,
        check_training_eligibility,
    )

    bundle = rd.slice_bundle()
    config = rd.config()
    target_cycle = bundle.cycles[-1]
    training_cutoff = datetime.combine(target_cycle.cutoff, time.min)

    dataset = build_training_dataset(
        target_cycle.quarter_start, training_cutoff, bundle.labeled_quarters,
        strategy_id=config.strategy_id, feature_schema_version=FEATURE_SCHEMA_VERSION,
        ml_train_years=config.ml_train_years, model_config_identity=config.model.identity(),
    )
    eligibility = check_training_eligibility(dataset, min_train_quarters=config.min_train_quarters)
    training_result = train_model(
        dataset, eligibility, config.model, build_lgbm_ranker_estimator,
        strategy_id=config.strategy_id, strategy_version=STRATEGY_VERSION,
    )
    print("target cycle  ", target_cycle.quarter_start, "(cutoff", target_cycle.cutoff, ")")
    print("training state", training_result.state.value)
else:
    bundle = None
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Score the real cross-section

No `group` is passed at predict time: LambdaRank grouping matters only for the training loss. `predict` scores each row independently and the caller compares scores within one quarter's batch.

Scoring selects exactly the columns the estimator was actually fit on — scoring on a different column set would be meaningless, not merely a shape mismatch.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.multi_factor_ranking_ml.scoring import score_observations

    observations = bundle.feature_results[target_cycle.quarter_start].observations
    scoring_result = score_observations(
        training_result.fitted_estimator,
        training_result.model_identity,
        observations,
        training_cutoff,
        config_identity=config.identity(),
    )
    print("input observations", f"{scoring_result.input_observation_count:,}")
    print("scored candidates ", f"{len(scoring_result.scored_candidates):,}")
    print("rejected          ", f"{len(scoring_result.rejected):,}")
    for rejection in scoring_result.rejected[:5]:
        print("   ", rejection.instrument_id.symbol, "|", rejection.reason[:100])
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Scores are unbounded ranker margins, not probabilities

Asserted directly on the real scores: they are not confined to `[0, 1]`, they do not sum to anything meaningful, and they take negative values.

In [ ]:
if AVAILABLE:
    import statistics

    scores = [candidate.score for candidate in scoring_result.scored_candidates]
    print(f"n        {len(scores):,}")
    print(f"min      {min(scores):.6f}")
    print(f"max      {max(scores):.6f}")
    print(f"mean     {statistics.mean(scores):.6f}")
    print(f"stdev    {statistics.pstdev(scores):.6f}")
    print(f"negative {sum(1 for s in scores if s < 0):,}")
    print(f"above 1  {sum(1 for s in scores if s > 1.0):,}")
    print()
    outside_unit_interval = [s for s in scores if not 0.0 <= s <= 1.0]
    assert outside_unit_interval, "a LambdaRank margin is not a probability"
    print(f"OK: {len(outside_unit_interval):,} of {len(scores):,} scores fall outside [0, 1].")
    print("    These are ranking margins. Only their ORDER carries meaning.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## The decision pipeline

Three steps, each returning `(kept, rejected)` so nothing is dropped without a recorded reason:

1. `validate_candidates` — structural validation and duplicate rejection. Scores must be *finite*, deliberately **not** bounded to `[0, 1]`: that bound was valid only while the score was a classifier probability and was deleted rather than widened to another arbitrary interval.
2. `apply_sector_exclusion` — exact, case-sensitive match against `config.exclude_sectors`.
3. `rank_candidates` / `to_ranked` — descending score, ties broken by ascending symbol, then 1-based ranks.

In [ ]:
if AVAILABLE:
    from collections import Counter

    from atlas_quant.strategies.multi_factor_ranking_ml.decision_pipeline import (
        apply_sector_exclusion,
        rank_candidates,
        to_ranked,
        validate_candidates,
    )

    candidates = scoring_result.scored_candidates

    valid, invalid = validate_candidates(
        candidates,
        strategy_id=config.strategy_id,
        feature_schema_version=FEATURE_SCHEMA_VERSION,
        evaluation_timestamp=training_cutoff,
    )
    print(f"1. validate         {len(candidates):,} -> {len(valid):,}  "
          f"({len(invalid):,} rejected)")
    for category, count in Counter(r.category.value for r in invalid).most_common():
        print(f"     {category:<28} {count:>5,}")

    kept, sector_rejected = apply_sector_exclusion(valid, config.exclude_sectors)
    print(f"2. sector exclusion {len(valid):,} -> {len(kept):,}  "
          f"({len(sector_rejected):,} in {config.exclude_sectors})")

    ranked = to_ranked(rank_candidates(kept))
    print(f"3. rank             {len(kept):,} -> {len(ranked):,} ranked instruments")
else:
    ranked = ()
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Ranks are contiguous over the whole cross-section

Not a top-N list: every surviving instrument gets a rank, 1..N with no gaps and no ties.

In [ ]:
if AVAILABLE:
    ranks = [r.rank for r in ranked]
    assert ranks == list(range(1, len(ranked) + 1))
    assert len({r.instrument_id for r in ranked}) == len(ranked)
    scores_in_rank_order = [r.score for r in ranked]
    assert scores_in_rank_order == sorted(scores_in_rank_order, reverse=True)
    print(f"OK: ranks are exactly 1..{len(ranked)}, contiguous, one per instrument,")
    print("    strictly ordered by descending score.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## The ranking itself

In [ ]:
if AVAILABLE:
    print(f"Multi-Factor Ranking ML — quarter {target_cycle.quarter_start}, "
          f"{len(ranked)} instruments ranked")
    print()
    print(f"{'rank':>6}  {'symbol':<12} {'score':>12}")
    print("  --- top 10 ---")
    for r in ranked[:10]:
        print(f"{r.rank:>6}  {r.instrument_id.symbol:<12} {r.score:>12.6f}")
    print("  --- middle ---")
    middle = len(ranked) // 2
    for r in ranked[middle - 2:middle + 3]:
        print(f"{r.rank:>6}  {r.instrument_id.symbol:<12} {r.score:>12.6f}")
    print("  --- bottom 10 ---")
    for r in ranked[-10:]:
        print(f"{r.rank:>6}  {r.instrument_id.symbol:<12} {r.score:>12.6f}")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Via the strategy evaluator

The same sequence, through `MultiFactorRankingMLStrategy.evaluate()` — the boundary the live runner and the backtest both go through. `capital_requested_pct` is always `0.0` and every recommendation's `weight` is `0.0`: those fields exist only to satisfy the shared platform contract, and this strategy allocates nothing.

In [ ]:
if AVAILABLE:
    from atlas_quant.strategies.base import StrategyEvaluationContext
    from atlas_quant.strategies.multi_factor_ranking_ml.strategy import (
        MultiFactorRankingEvaluationInputs,
        MultiFactorRankingMLStrategy,
    )

    context = StrategyEvaluationContext(
        strategy_id=config.strategy_id,
        evaluation_timestamp=training_cutoff,
        data_cutoff=training_cutoff,
        capital_budget_pct=0.0,
        strategy_config=MultiFactorRankingEvaluationInputs(
            config=config, scored_candidates=scoring_result.scored_candidates
        ),
    )
    result = MultiFactorRankingMLStrategy().evaluate(context)
    summary = result.state_update

    print("status                ", result.status.value)
    print("outcome               ", summary.outcome.value)
    print("initial candidates    ", f"{summary.initial_candidate_count:,}")
    print("rejected              ", f"{len(summary.rejected_candidates):,}")
    print("ranked                ", f"{len(summary.ranked_candidates):,}")
    print("capital_requested_pct ", result.capital_requested_pct)
    print("distinct weights      ", {rec.weight for rec in result.recommendations})
    assert [r.rank for r in summary.ranked_candidates] == [r.rank for r in ranked]
    print()
    print("OK: the evaluator reproduces the step-by-step pipeline above exactly.")
else:
    print(rd.DATA_UNAVAILABLE_MESSAGE)


## Findings

- The real cross-section scores to unbounded margins spanning negative and greater-than-one values — confirming directly that these are LambdaRank margins, not the calibrated probabilities the previous classifier emitted.
- The decision pipeline rejects nothing structurally on real data; the only material attrition is the Materials sector exclusion.
- The output is a contiguous 1..N ranking of the entire surviving universe (~1,400 names), asserted to be gap-free, tie-free and strictly score-ordered. That full ranking *is* the strategy's product.

## Limitations

- A rank is an ordering claim, not an expected return. Nothing here says how much better rank 1 is than rank 2, or that the ordering is profitable — that is what the IC evaluation in notebooks 08–09 is for.
- Scores are comparable only *within* this one scoring batch. Comparing a score from this quarter against one from another quarter's separately-fitted model is meaningless.
- The sector exclusion uses the present-day GICS snapshot, so a name is excluded (or not) on the basis of its current classification rather than its classification at the time.